# Audio Inpainting with PlayDiffusion — v5.0

Run Cell 1 (install) — wait for it to finish, then restart runtime when prompted.
Run Cell 1 again (it will skip installs and just confirm everything is ready).
Then run cells 4 through 20 sequentially.

This notebook runs the full audio inpainting pipeline on Google Colab:
1. **Setup** — clone PlayDiffusion, install PyTorch 2.6.0 + fairseq2 + dependencies
2. **STT + LLM** — transcribe corrupted audio with Qwen3-ASR (or Whisper), fix gaps with Qwen3-32B (OpenRouter)
3. **Detection** — four complementary corruption-detection methods:
   - Method A: RMS Energy Dropout (signal processing)
   - Method B: MFCC Discontinuity with adaptive threshold (signal processing)
   - Method C: WavLM Frame-Embedding Discontinuity (neural SSL)
   - Method D: Silero VAD (neural VAD) [NEW in v5.0]
4. **Benchmark** — compare all four methods against ground-truth event files [NEW in v5.0]
5. **Inpainting** — use PlayDiffusion to regenerate corrupted regions

**v5.0 changes vs v4.0:**
- Added Silero VAD (Method D) — a dedicated neural VAD model for dropout detection
- Added standalone detection benchmark cell (Step 7a) with a per-method comparison table
- Updated merge logic to incorporate Silero VAD results
- Qwen2-Audio kept as a stub but remains disabled (poor timestamp accuracy on this task)

**Requirements:** Colab GPU runtime. T4 (16 GB) is sufficient for all methods in v5.0.

In [ ]:
import os

# ── 1. Clone PlayDiffusion ──────────────────────────────────────────────
if not os.path.exists('/content/PlayDiffusion'):
    !git clone https://github.com/playht/PlayDiffusion.git
else:
    print("PlayDiffusion already cloned — skipping.")

# ── 2. Check whether dependencies are already installed (post-restart) ──
_need_install = True
try:
    import torch
    if torch.__version__.startswith('2.6.0'):
        import torchaudio, fairseq2
        _need_install = False
        print("Dependencies already installed. Skip to the next cell.")
except ImportError:
    pass

if _need_install:
    print("Installing dependencies — this may take a few minutes ...\n")

    # Remove Colab-default packages that conflict with PlayDiffusion
    !pip uninstall -y torch torchvision torchaudio fairseq2 fairseq2n 2>/dev/null

    # Step A: Install PyTorch 2.6.0 first
    !pip install torch==2.6.0 torchaudio==2.6.0 \
        --index-url https://download.pytorch.org/whl/cu124

    # Step B: Install fairseq2 explicitly using the PyTorch 2.6.0 + CUDA 12.4 index
    # Enforce torch==2.6.0 so fairseq2n doesn't pull a newer version
    !pip install fairseq2==0.4.4 fairseq2n==0.4.4 torch==2.6.0 torchaudio==2.6.0 \
        --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.6.0/cu124

    # Step C: Install PlayDiffusion runtime deps
    !pip install \
        "numpy>=1.26,<2.1" \
        "scipy>=1.12" \
        "scikit-learn>=1.3" \
        nltk jiwer pydantic soundfile boto3 tqdm python-decouple \
        safetensors tokenizers librosa einops huggingface-hub unidecode \
        torchtune==0.6.1 torchao==0.11.0 torch==2.6.0 torchaudio==2.6.0
    !pip install "syllables @ git+https://github.com/playht/python-syllables.git"

    # [NEW] Ensure transformers is new enough for Qwen2-Audio-7B-Instruct
    !pip install -q "transformers==4.45.2"

    # Step D: Install Whisper (STT) + additional signal/eval deps
    !pip install openai-whisper whisper-timestamped google-generativeai torch==2.6.0 torchaudio==2.6.0

    # Step E: Silero VAD (Method D) + audio quality eval
    !pip install -q silero-vad pystoi

    print("\n" + "=" * 60)
    print("  RESTART THE RUNTIME NOW")
    print("  Runtime -> Restart runtime, then run the cells below.")
    print("=" * 60)

### After running the cell above for the first time:
1. **Restart the runtime** — Runtime -> Restart runtime
2. Then continue running from the cell below (you do **not** need to re-run the install cell)

### Step 2: Verify Installation

In [ ]:
import sys
sys.path.insert(0, '/content/PlayDiffusion/src')

import torch, torchaudio
print(f"Python:     {sys.version.split()[0]}")
print(f"PyTorch:    {torch.__version__}")
print(f"Torchaudio: {torchaudio.__version__}")
print(f"CUDA:       {torch.cuda.is_available()}  (toolkit {torch.version.cuda})")

import fairseq2
print(f"fairseq2:   {fairseq2.__version__}")

from playdiffusion import PlayDiffusion, InpaintInput
print("\nPlayDiffusion imported successfully!")

### Step 3: Prepare Data
Mount Google Drive and set the input/output directories. Upload your preprocessed audio files and JSON metadata to the input directory.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Point these at the folder where your preprocessed audio + JSON metadata live.
# Option A (Google Drive — persists between sessions):

# input_dir  = "/content/drive/MyDrive/samples/preprocessed_audio"
# output_dir = "/content/drive/MyDrive/samples/inpainted_audio"

input_dir  = "/content/drive/MyDrive/samples_hindy/preprocessed"
output_dir = "/content/drive/MyDrive/samples_hindy/inpainted_audio"


os.makedirs(input_dir,  exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print(f"Input dir:  {input_dir}")
print(f"Output dir: {output_dir}")

### Step 4: Load Models & Helper Functions

This cell loads:
*   Qwen3-ASR-1.7B + ForcedAligner-0.6B — for speech-to-text with native word-level timestamps (replaces Whisper)
* Qwen3-32B via OpenRouter — for LLM transcript correction

[CHANGED] ASR switched from Whisper → Qwen3-ASR (SOTA open-source, better word boundaries, native forced alignment). Whisper is kept as fallback in case Qwen3-ASR fails to load (e.g. dependency / memory issue).

Helper functions defined here (used by Steps 5 and 6):
* transcribe_with_timestamps()  — unified interface (Qwen or Whisper)
* get_corrected_text_and_original_word_times()
* has_acoustic_discontinuity()
* smooth_silence_gap()


In [ ]:
import numpy as np
import librosa
import soundfile as sf
import requests
import torch
from google.colab import userdata

# ── [NEW] Load Qwen3-ASR (primary), fall back to Whisper if it fails ──────────
USE_QWEN_ASR = False          # Set False to force Whisper instead
QWEN_ASR_MODEL_ID = "Qwen/Qwen3-ASR-1.7B"      # Use "Qwen3-ASR-0.6B" if OOM
QWEN_ALIGNER_ID   = "Qwen/Qwen3-ForcedAligner-0.6B"

qwen_asr_model = None
whisper_model  = None

if USE_QWEN_ASR:
    try:
        print(f"Loading {QWEN_ASR_MODEL_ID} + forced aligner ...")
        from qwen_asr import Qwen3ASRModel
        qwen_asr_model = Qwen3ASRModel.from_pretrained(
            QWEN_ASR_MODEL_ID,
            dtype=torch.bfloat16,
            device_map="cuda:0",
            max_inference_batch_size=8,
            max_new_tokens=512,
            forced_aligner=QWEN_ALIGNER_ID,
            forced_aligner_kwargs=dict(
                dtype=torch.bfloat16,
                device_map="cuda:0",
            ),
        )
        print("Qwen3-ASR loaded.")
    except Exception as e:
        print(f"Qwen3-ASR failed to load ({e}) — falling back to Whisper.")
        qwen_asr_model = None

if qwen_asr_model is None:
    print("Loading Whisper model (fallback) ...")
    import whisper
    import whisper_timestamped as whisper_ts
    whisper_model = whisper.load_model("base")

# ── Configure Qwen via OpenRouter (free LLM for transcript correction) ────────
OPENROUTER_API_KEY = None
try:
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    print("Qwen (OpenRouter) configured.")
except Exception as e:
    print(f"Could not load OPENROUTER_API_KEY: {e}")
    print("Running without LLM correction — raw ASR transcripts will be used.")


def call_qwen_llm(transcript):
    """Sends a broken transcript to Qwen3-32B via OpenRouter for correction."""
    if OPENROUTER_API_KEY is None:
        return None
    prompt = (
        "The following text is a transcription of an audio file that had "
        "network connectivity drops. There are missing words or parts of words. "
        "Please guess what the full, logical sentence should be and return "
        "ONLY the complete, corrected sentence without any explanation.\n\n"
        f"Transcript: {transcript}"
    )
    try:
        response = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json",
            },
            json={
                "model": "qwen/qwen3-32b:free",
                "messages": [{"role": "user", "content": prompt}],
            },
            timeout=30,
        )
        return response.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"  LLM error ({e}) — using raw transcript.")
        return None


# ── [NEW] Unified transcription function ──────────────────────────────────────
# Returns (transcript, word_timestamps) where word_timestamps is a list of
# {'word', 'start', 'end'} dicts — same format as before, regardless of
# whether Qwen3-ASR or Whisper was used underneath.
def transcribe_with_timestamps(audio_path):
    if qwen_asr_model is not None:
        # Qwen3-ASR path
        results = qwen_asr_model.transcribe(
            audio=audio_path,
            language="English",
            return_time_stamps=True,
        )
        r = results[0]
        transcript = r.text

        word_timestamps = []
        ts = getattr(r, 'time_stamps', None) or []
        # time_stamps can be a flat list of word objects or a list-of-lists;
        # handle both to be safe across package versions
        flat = ts[0] if (ts and isinstance(ts[0], (list, tuple))) else ts
        for w in flat:
            word = getattr(w, 'text', None) or (w.get('text') if isinstance(w, dict) else '')
            s    = getattr(w, 'start_time', None) or (w.get('start_time') if isinstance(w, dict) else 0.0)
            e    = getattr(w, 'end_time',   None) or (w.get('end_time')   if isinstance(w, dict) else 0.0)
            word_timestamps.append({'word': word.strip(), 'start': float(s), 'end': float(e)})
        return transcript, word_timestamps
    else:
        # Whisper fallback path
        audio     = whisper_ts.load_audio(audio_path)
        result_ts = whisper_ts.transcribe(whisper_model, audio, language="en", verbose=False)
        transcript = result_ts['text']
        word_timestamps = []
        for seg in result_ts.get('segments', []):
            for wi in seg.get('words', []):
                word_timestamps.append({
                    'word':  wi.get('text', '').strip(),
                    'start': wi['start'],
                    'end':   wi['end'],
                })
        return transcript, word_timestamps


def get_corrected_text_and_original_word_times(audio_path):
    """Transcribes audio, then corrects the transcript with the LLM."""
    print(f"  Transcribing ...")
    transcript, word_timestamps = transcribe_with_timestamps(audio_path)
    print(f"  Original:  {transcript}")

    corrected = call_qwen_llm(transcript)
    if corrected:
        print(f"  Corrected: {corrected}")
        return corrected, word_timestamps
    return transcript, word_timestamps


# ── Helper: MFCC discontinuity check at a known cut point ─────────────────────
def has_acoustic_discontinuity(audio_path, cut_time_sec, window_sec=0.05, threshold=10.0):
    y, sr      = librosa.load(audio_path, sr=None, mono=True)
    cut_sample = int(cut_time_sec * sr)
    window     = int(window_sec   * sr)
    before = y[max(0, cut_sample - window) : cut_sample]
    after  = y[cut_sample : min(len(y), cut_sample + window)]
    if len(before) < 10 or len(after) < 10:
        return False
    m_before = librosa.feature.mfcc(y=before, sr=sr, n_mfcc=13)
    m_after  = librosa.feature.mfcc(y=after,  sr=sr, n_mfcc=13)
    diff = np.linalg.norm(np.mean(m_before, axis=1) - np.mean(m_after, axis=1))
    return bool(diff > threshold)


# ── Helper: smooth a silence gap with a short crossfade ───────────────────────
def smooth_silence_gap(audio_path, start_sec, end_sec, output_path, crossfade_ms=20):
    y, sr = sf.read(audio_path)
    mono  = (y.ndim == 1)
    if mono:
        y = y[:, np.newaxis]
    s_sample = int(start_sec * sr)
    e_sample = int(end_sec   * sr)
    cf = min(int(crossfade_ms * sr / 1000), s_sample, max(0, len(y) - e_sample))
    if cf > 0:
        fade_out = np.linspace(1.0, 0.0, cf)[:, np.newaxis]
        fade_in  = np.linspace(0.0, 1.0, cf)[:, np.newaxis]
        blended  = y[s_sample - cf : s_sample] * fade_out + y[e_sample : e_sample + cf] * fade_in
        result   = np.concatenate([y[: s_sample - cf], blended, y[e_sample + cf :]])
    else:
        result = np.concatenate([y[:s_sample], y[e_sample:]])
    if mono:
        result = result[:, 0]
    sf.write(output_path, result, sr)

### Step 5: Corruption Detection Module

Four complementary methods detect corrupted audio regions. They are run independently and their results merged.

**Method A — RMS Energy Dropout**
Computes rolling loudness (dB) and flags sudden drops. Best at: full dropouts, hard silence insertions.

**Method B — MFCC Discontinuity Scan**
Computes frame-level acoustic feature distance with an adaptive threshold (median + k×MAD, floored at p99). Best at: mid-word glitches, cut-and-splice artifacts.

**Method C — WavLM Frame-Embedding Discontinuity**
Computes cosine distance on 768-dim WavLM embeddings at 50 Hz. Catches phonetic discontinuities invisible to spectral methods. Best at: mid-word splices where MFCC shape stays similar but phonetic content jumps.

**Method D — Silero VAD** *(NEW in v5.0)*
A dedicated neural VAD model (~8 MB). Finds gaps between speech segments — non-speech regions within a speech recording indicate network dropouts. Best at: clean silence dropouts, robust to codec/channel noise.

All methods run independently. Results are merged into a unified event list.

In [ ]:
import numpy as np
import librosa

# ── Method A — RMS dropout detection ──────────────────────────────────────────
RMS_WINDOW_SEC      = 0.020   # Analysis window size (20ms)
RMS_HOP_SEC         = 0.010   # Hop between windows (10ms)
RMS_DROP_THRESHOLD  = 20.0    # dB drop vs local max that flags a dropout
RMS_MIN_DURATION    = 0.020   # Ignore detections shorter than this (seconds)

# ── Method B — MFCC discontinuity scan ────────────────────────────────────────
MFCC_FRAME_SEC       = 0.025   # MFCC analysis window (25ms)
MFCC_HOP_SEC         = 0.010   # MFCC hop (10ms)
MFCC_N_COEFFS        = 13      # Number of MFCC coefficients
MFCC_OUTLIER_K       = 6.0     # Robust z-score multiplier (median + k*MAD)
MFCC_MIN_PERCENTILE  = 99.0    # Never trigger below this percentile

# ── Merging and padding ───────────────────────────────────────────────────────
MERGE_GAP_SEC       = 0.05    # Fuse detections closer than this (seconds)
CONTEXT_PAD_SEC     = 0.05    # Expand each detected region by this on each side
MAX_REGION_SEC      = 2.5     # Sanity cap on absurdly large fused regions


def detect_rms_dropouts(y, sr):
    win = int(RMS_WINDOW_SEC * sr)
    hop = int(RMS_HOP_SEC    * sr)
    eps = 1e-10

    frames = librosa.util.frame(y, frame_length=win, hop_length=hop)
    rms    = np.sqrt(np.mean(frames ** 2, axis=0))
    rms_db = 20 * np.log10(rms + eps)
    times  = librosa.frames_to_time(np.arange(len(rms_db)), sr=sr, hop_length=hop)

    ref_frames  = max(1, int(0.3 / RMS_HOP_SEC))
    rolling_max = np.array([
        np.max(rms_db[max(0, i - ref_frames) : i + 1])
        for i in range(len(rms_db))
    ])
    dropout_mask = (rolling_max - rms_db) > RMS_DROP_THRESHOLD

    events     = []
    in_dropout = False
    start_idx  = 0
    for i, flag in enumerate(dropout_mask):
        if flag and not in_dropout:
            in_dropout = True
            start_idx  = i
        elif not flag and in_dropout:
            in_dropout = False
            if (times[i] - times[start_idx]) >= RMS_MIN_DURATION:
                events.append((times[start_idx], times[i]))
    if in_dropout and (times[-1] - times[start_idx]) >= RMS_MIN_DURATION:
        events.append((times[start_idx], times[-1]))

    return events


def detect_mfcc_discontinuities(y, sr):
    hop   = int(MFCC_HOP_SEC   * sr)
    n_fft = int(MFCC_FRAME_SEC * sr)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=MFCC_N_COEFFS,
                                  n_fft=n_fft, hop_length=hop)
    diffs = np.linalg.norm(np.diff(mfccs, axis=1), axis=0)
    times = librosa.frames_to_time(np.arange(len(diffs)), sr=sr, hop_length=hop)

    med = float(np.median(diffs))
    mad = float(np.median(np.abs(diffs - med)) * 1.4826 + 1e-9)
    thr = max(med + MFCC_OUTLIER_K * mad,
              float(np.percentile(diffs, MFCC_MIN_PERCENTILE)))

    n_above = int((diffs > thr).sum())
    print(f'  [MFCC stats] median={med:.2f}  mad={mad:.2f}  '
          f'thr={thr:.2f}  n_above={n_above}')

    return [(float(times[i]), float(times[i])) for i, d in enumerate(diffs) if d > thr]


def merge_events_with_source(rms_events, mfcc_events, wavlm_events=None,
                              silero_events=None, audio_duration=None):
    wavlm_events  = wavlm_events  or []
    silero_events = silero_events or []
    tagged = ([(s, e, 'rms')    for s, e in rms_events]
              + [(s, e, 'mfcc')   for s, e in mfcc_events]
              + [(s, e, 'wavlm')  for s, e in wavlm_events]
              + [(s, e, 'silero') for s, e in silero_events])
    if not tagged:
        return []
    tagged.sort(key=lambda x: x[0])

    merged = [[tagged[0][0], tagged[0][1], {tagged[0][2]}]]
    for s, e, src in tagged[1:]:
        if s - merged[-1][1] <= MERGE_GAP_SEC:
            merged[-1][1] = max(merged[-1][1], e)
            merged[-1][2].add(src)
        else:
            merged.append([s, e, {src}])

    result = []
    for s, e, srcs in merged:
        s = max(0.0, s - CONTEXT_PAD_SEC)
        e = e + CONTEXT_PAD_SEC
        if audio_duration:
            e = min(audio_duration, e)
        result.append((s, e, srcs))

    kept, dropped = [], []
    for s, e, srcs in result:
        (kept if (e - s) <= MAX_REGION_SEC else dropped).append((s, e, srcs))
    if dropped:
        print(f'  [WARN] dropped {len(dropped)} oversized region(s) (>{MAX_REGION_SEC}s)')

    return kept


def detect_corrupted_regions(audio_path, verbose=True, mode='any'):
    """
    Runs all available detectors and merges results.
    Modes: 'any', 'both' (RMS AND MFCC), 'rms', 'mfcc', 'wavlm', 'silero',
           'wavlm+rms', 'wavlm+silero' (both neural methods).
    """
    y, sr    = librosa.load(audio_path, sr=None, mono=True)
    duration = len(y) / sr

    rms_events  = detect_rms_dropouts(y, sr)
    mfcc_events = detect_mfcc_discontinuities(y, sr)

    try:
        wavlm_events = detect_wavlm_discontinuities(audio_path, verbose=verbose)
    except NameError:
        wavlm_events = []

    try:
        silero_events = detect_silero_dropouts(audio_path, verbose=verbose)
    except NameError:
        silero_events = []

    if verbose:
        print(f'  [Method A — RMS]    {len(rms_events)} candidate(s)')
        print(f'  [Method B — MFCC]   {len(mfcc_events)} candidate(s)')
        print(f'  [Method C — WavLM]  {len(wavlm_events)} candidate(s)')
        print(f'  [Method D — Silero] {len(silero_events)} candidate(s)')

    merged = merge_events_with_source(rms_events, mfcc_events, wavlm_events,
                                      silero_events, audio_duration=duration)

    if mode == 'both':
        merged = [(s, e, srcs) for s, e, srcs in merged if {'rms', 'mfcc'}.issubset(srcs)]
    elif mode == 'rms':
        merged = [(s, e, srcs) for s, e, srcs in merged if 'rms' in srcs]
    elif mode == 'mfcc':
        merged = [(s, e, srcs) for s, e, srcs in merged if 'mfcc' in srcs]
    elif mode == 'wavlm':
        merged = [(s, e, srcs) for s, e, srcs in merged if 'wavlm' in srcs]
    elif mode == 'silero':
        merged = [(s, e, srcs) for s, e, srcs in merged if 'silero' in srcs]
    elif mode == 'wavlm+rms':
        merged = [(s, e, srcs) for s, e, srcs in merged if {'wavlm', 'rms'}.issubset(srcs)]
    elif mode == 'wavlm+silero':
        merged = [(s, e, srcs) for s, e, srcs in merged if {'wavlm', 'silero'}.issubset(srcs)]
    # mode == 'any' keeps everything

    if verbose:
        print(f'  [Merged, mode={mode}] {len(merged)} region(s):')
        for s, e, srcs in merged:
            print(f'                    {s:.3f}s - {e:.3f}s  [{chr(43).join(sorted(srcs))}]')

    return [{'start_sec': s, 'end_sec': e, 'source': '+'.join(sorted(srcs))}
            for s, e, srcs in merged]


print('Step 5 ready: Corruption detection module loaded.')
print(f'  RMS drop threshold:  {RMS_DROP_THRESHOLD} dB')
print(f'  MFCC adaptive:       median + {MFCC_OUTLIER_K}*MAD, floor=p{MFCC_MIN_PERCENTILE}')
print(f'  Merge gap:           {MERGE_GAP_SEC}s')
print(f'  Max region:          {MAX_REGION_SEC}s (sanity cap)')


### Step 5b: Qwen2-Audio Detection (Disabled)

Qwen2-Audio-7B-Instruct was evaluated in v3.0 but produced imprecise timestamps and unreliable detections on short telephony dropouts. It remains as a stub in case future prompt engineering improves it, but is disabled by default.

Silero VAD (Step 5d) is the replacement — a dedicated neural VAD model that is lighter weight and produces reliable timestamps.

In [ ]:
# [CHANGED] Qwen2-Audio disabled — confirmed wrong tool for short-dropout detection.
# It hallucinated timestamps under open-ended prompts and returned empty under
# structured prompts. Keeping the function stub so Step 7a code paths still work.
USE_QWEN_AUDIO   = False
qwen_audio_model = None

def detect_with_qwen_audio(audio_path, verbose=True, debug=False):
    return []

print("Step 5b: Qwen2-Audio disabled (low-quality outputs on this task).")

In [ ]:
# ── Step 5c: WavLM Frame-Embedding Discontinuity Detection [NEW] ─────────────
#
# Computes WavLM frame embeddings (50Hz / 20ms hops) and flags frames where the
# cosine distance to the previous frame is an outlier. Catches phonetic
# discontinuities that MFCC misses — specifically the "mid-word splice" case
# where spectral shape stays similar but phonetic content jumps abruptly.
#
# Model: microsoft/wavlm-base-plus (95M, ~360MB). Loads to GPU, inference only.
# ─────────────────────────────────────────────────────────────────────────────

import torch
import numpy as np
import librosa

USE_WAVLM = True

wavlm_model     = None
wavlm_processor = None

if USE_WAVLM:
    try:
        print("Loading WavLM-base-plus ...")
        from transformers import AutoFeatureExtractor, WavLMModel
        wavlm_processor = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
        wavlm_model     = WavLMModel.from_pretrained(
            "microsoft/wavlm-base-plus",
            torch_dtype=torch.float32,   # WavLM is small enough; fp32 keeps cosine distances stable
        ).to("cuda:0").eval()
        print("WavLM loaded.")
    except Exception as e:
        print(f"WavLM failed to load ({e}) — skipping this method.")
        wavlm_model = None


# ── Detection params ──────────────────────────────────────────────────────────
WAVLM_OUTLIER_K      = 3.0    # [CHANGED] was 5.0; cosine MAD is large so 5*MAD over-shot every distance
WAVLM_MIN_PERCENTILE = 97.0   # [CHANGED] was 99.0; ~p99 was 4-5 frames on 9s clips, too sparse to gate on
WAVLM_HOP_SEC        = 0.020  # WavLM-base outputs at 50Hz (one frame per 20ms)


def _wavlm_frame_embeddings(audio_path):
    """
    Returns (embeddings [T, D], frame_times [T]) for the audio.
    WavLM-base outputs 768-dim embeddings at 50Hz.
    """
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    inputs = wavlm_processor(audio, sampling_rate=16000, return_tensors="pt")
    inputs = {k: v.to(wavlm_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = wavlm_model(**inputs)
    # last_hidden_state: [1, T, 768]
    emb = out.last_hidden_state[0].float().cpu().numpy()
    times = np.arange(emb.shape[0]) * WAVLM_HOP_SEC
    return emb, times


def detect_wavlm_discontinuities(audio_path, verbose=True):
    """
    [NEW] Frame-to-frame cosine distance on WavLM embeddings, flagged with
    the same robust adaptive threshold we use for MFCC. Returns zero-width
    events; merge_events_with_source pads them.
    """
    if wavlm_model is None:
        return []

    emb, times = _wavlm_frame_embeddings(audio_path)

    # Cosine distance between consecutive frames
    norms = np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9
    unit  = emb / norms
    cos_sim = np.sum(unit[1:] * unit[:-1], axis=1)
    dists   = 1.0 - cos_sim                     # length T-1
    diff_times = times[1:]

    med = float(np.median(dists))
    mad = float(np.median(np.abs(dists - med)) * 1.4826 + 1e-9)
    thr = max(med + WAVLM_OUTLIER_K * mad,
              float(np.percentile(dists, WAVLM_MIN_PERCENTILE)))

    if verbose:
      p95 = float(np.percentile(dists, 95))
      p99 = float(np.percentile(dists, 99))
      p_max = float(dists.max())
      print(f"  [WavLM stats] median={med:.4f}  mad={mad:.4f}  "
            f"p95={p95:.4f}  p99={p99:.4f}  max={p_max:.4f}  "
            f"thr={thr:.4f}  n_above={int((dists > thr).sum())}")

    return [(float(diff_times[i]), float(diff_times[i]))
            for i, d in enumerate(dists) if d > thr]


print("Step 5c ready.")
print(f"  WavLM available: {wavlm_model is not None}")

In [ ]:
# ── Step 5d: Silero VAD Dropout Detection [NEW in v5.0] ─────────────────────
#
# Silero VAD is a lightweight (~8 MB) neural model trained to detect speech vs.
# non-speech. For dropout detection we invert it: gaps between speech segments
# within a recording indicate network dropouts.
#
# Complementary to WavLM: Silero catches silence dropouts (hard zeros, RFC 3389
# comfort noise) while WavLM catches phonetic splices that remain non-silent.
# ─────────────────────────────────────────────────────────────────────────────

USE_SILERO_VAD = True

silero_vad_model      = None
_silero_get_speech_ts = None
_silero_read_audio    = None

if USE_SILERO_VAD:
    try:
        print('Loading Silero VAD ...')
        from silero_vad import load_silero_vad, read_audio, get_speech_timestamps
        silero_vad_model      = load_silero_vad()
        _silero_get_speech_ts = get_speech_timestamps
        _silero_read_audio    = read_audio
        print('Silero VAD loaded.')
    except Exception as e:
        print(f'Silero VAD failed to load ({e}) — skipping this method.')
        silero_vad_model = None

# ── Detection parameters ──────────────────────────────────────────────────────
SILERO_THRESHOLD      = 0.5    # Speech prob below which a frame is 'non-speech'
SILERO_MIN_SPEECH_MS  = 100    # Ignore isolated speech islands shorter than this
SILERO_MIN_SILENCE_MS = 30     # Minimum non-speech gap to report
SILERO_MIN_GAP_SEC    = 0.020  # Minimum gap (sec) to flag as a dropout candidate


def detect_silero_dropouts(audio_path, verbose=True):
    """
    [Method D] Runs Silero VAD and returns gaps between speech segments as
    (start_sec, end_sec) dropout candidates.

    Gaps at the very start or end of the file are excluded (natural silence).
    """
    if silero_vad_model is None:
        return []

    wav = _silero_read_audio(audio_path, sampling_rate=16000)
    speech_ts = _silero_get_speech_ts(
        wav, silero_vad_model,
        sampling_rate=16000,
        threshold=SILERO_THRESHOLD,
        min_speech_duration_ms=SILERO_MIN_SPEECH_MS,
        min_silence_duration_ms=SILERO_MIN_SILENCE_MS,
        return_seconds=True,
    )

    # Gaps BETWEEN speech segments (not at file edges) are dropout candidates
    dropouts = []
    for i in range(len(speech_ts) - 1):
        gap_start = speech_ts[i]['end']
        gap_end   = speech_ts[i + 1]['start']
        if (gap_end - gap_start) >= SILERO_MIN_GAP_SEC:
            dropouts.append((gap_start, gap_end))

    if verbose:
        print(f'  [Method D — Silero VAD] {len(dropouts)} candidate(s):')
        for s, e in dropouts:
            print(f'                         {s:.3f}s - {e:.3f}s')

    return dropouts


print('Step 5d ready.')
print(f'  Silero VAD available:  {silero_vad_model is not None}')
print(f'  Speech threshold:      {SILERO_THRESHOLD}')
print(f'  Min silence reported:  {SILERO_MIN_SILENCE_MS} ms')


### Step 6: Evaluation Helpers

[NEW] When JSON events are available, we treat them as GROUND TRUTH and run the auto-detection (Step 5) in parallel to measure how well detection is doing on real data.

Metrics computed per file and aggregated at the end of Step 7:
* Recall    = fraction of JSON events that were detected
* Precision = fraction of detected events that match a real JSON event
* Mean IoU  = average Intersection-over-Union for matched pairs

An event is "matched" if its time range overlaps with a JSON event with IoU >= MATCH_IOU_THRESHOLD.

The results of Step 7 still use the JSON events for actual inpainting (so the repair is as correct as possible); detection runs purely for measurement.


In [ ]:
import numpy as np

# Lowered from 0.5 — dropouts are short, so even small timing offsets can
# push a valid match below 0.5 IoU. 0.3 is more forgiving while still
# enforcing meaningful overlap.
MATCH_IOU_THRESHOLD = 0.3


def _iou(a, b):
    """Intersection-over-Union of two (start, end) intervals."""
    inter = max(0.0, min(a[1], b[1]) - max(a[0], b[0]))
    union = max(a[1], b[1]) - min(a[0], b[0])
    return inter / union if union > 0 else 0.0


def evaluate_detection(gt_events, detected_events):
    """
    Compares ground-truth JSON events to auto-detected events.
    Returns a dict with counts and recall / precision / mean_iou.

    Matching is greedy: for each GT event, pick the unmatched detection
    with the highest IoU. If that IoU >= MATCH_IOU_THRESHOLD, count as matched.
    """
    gt  = [(e["start_sec"], e["end_sec"]) for e in gt_events]
    det = [(e["start_sec"], e["end_sec"]) for e in detected_events]

    matched_gt  = set()
    matched_det = set()
    iou_values  = []

    for i, g in enumerate(gt):
        best_j, best_iou = -1, 0.0
        for j, d in enumerate(det):
            if j in matched_det:
                continue
            iou = _iou(g, d)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= MATCH_IOU_THRESHOLD:
            matched_gt.add(i)
            matched_det.add(best_j)
            iou_values.append(best_iou)

    recall    = len(matched_gt)  / len(gt)  if gt  else 1.0
    precision = len(matched_det) / len(det) if det else 1.0
    mean_iou  = float(np.mean(iou_values)) if iou_values else 0.0

    return {
        "n_gt":       len(gt),
        "n_detected": len(det),
        "n_matched":  len(matched_gt),
        "recall":     recall,
        "precision":  precision,
        "mean_iou":   mean_iou,
    }


def print_aggregate_metrics(totals):
    """Pretty-print the running totals collected across all processed files."""
    if totals["n_gt"] == 0:
        print("\nNo ground-truth events available — skipping aggregate metrics.")
        return

    overall_recall    = totals["n_matched"]  / totals["n_gt"]
    overall_precision = (totals["n_matched"] / totals["n_detected"]
                          if totals["n_detected"] else 0.0)
    overall_mean_iou  = (totals["iou_sum"] / totals["iou_count"]
                          if totals["iou_count"] else 0.0)

    print(f"\n{'=' * 60}")
    print("AGGREGATE DETECTION METRICS")
    print(f"{'=' * 60}")
    print(f"  Total GT events:       {totals['n_gt']}")
    print(f"  Total detected events: {totals['n_detected']}")
    print(f"  Total matched:         {totals['n_matched']}")
    print(f"  Overall recall:        {overall_recall:.2%}")
    print(f"  Overall precision:     {overall_precision:.2%}")
    print(f"  Mean IoU (matched):    {overall_mean_iou:.3f}")


print("Step 6 ready: evaluation helpers loaded.")
print(f"  IoU match threshold: {MATCH_IOU_THRESHOLD}")

### Step 7a: Detection Benchmark — All Methods vs Ground Truth

[NEW in v5.0] Runs all four detection methods independently on every sample in `input_dir` and compares against the JSON ground-truth events. No inpainting is performed — this is a pure detection quality report.

Use this cell to:
- Compare Method A (RMS), B (MFCC), C (WavLM), D (Silero VAD) side-by-side
- Choose the best `DETECTION_MODE` for the inpainting loop in Step 7
- Quickly iterate on detection thresholds without re-running PlayDiffusion

**Metrics** (IoU ≥ 0.3 match threshold):
- **Recall** — fraction of GT events detected
- **Precision** — fraction of detections that match a real event
- **F1** — harmonic mean of recall and precision
- **Mean IoU** — average overlap quality for matched pairs
- **Det/file** — average detections per file (tracks over-firing)

In [ ]:
# ── Step 7a: Detection Benchmark (no inpainting) ─────────────────────────────
#
# Runs all four detection methods independently on every sample in input_dir,
# compares against JSON ground-truth events, and prints a comparison table.
#
# Requires Steps 5, 5c, 5d, and 6 to be run first.
# ─────────────────────────────────────────────────────────────────────────────

import os, json, glob
import librosa
import numpy as np


def _pad_and_merge_single(raw_events, audio_duration):
    """Apply CONTEXT_PAD_SEC + merge to a single method's (start, end) list."""
    if not raw_events:
        return []
    padded = [(max(0.0, s - CONTEXT_PAD_SEC),
               min(audio_duration, e + CONTEXT_PAD_SEC))
              for s, e in raw_events]
    padded.sort()
    merged = [list(padded[0])]
    for s, e in padded[1:]:
        if s - merged[-1][1] <= MERGE_GAP_SEC:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return [{'start_sec': s, 'end_sec': e}
            for s, e in merged if (e - s) <= MAX_REGION_SEC]


def _empty_totals():
    return {'n_gt': 0, 'n_detected': 0, 'n_matched': 0,
            'iou_sum': 0.0, 'iou_count': 0}


def _accumulate(totals, metrics):
    totals['n_gt']       += metrics['n_gt']
    totals['n_detected'] += metrics['n_detected']
    totals['n_matched']  += metrics['n_matched']
    totals['iou_sum']    += metrics['mean_iou'] * metrics['n_matched']
    totals['iou_count']  += metrics['n_matched']


def _summary(totals):
    if totals['n_gt'] == 0:
        return {'recall': 0.0, 'precision': 0.0, 'f1': 0.0, 'mean_iou': 0.0}
    recall    = totals['n_matched'] / totals['n_gt']
    precision = (totals['n_matched'] / totals['n_detected']
                 if totals['n_detected'] else 0.0)
    f1        = (2 * recall * precision / (recall + precision)
                 if (recall + precision) > 0 else 0.0)
    mean_iou  = (totals['iou_sum'] / totals['iou_count']
                 if totals['iou_count'] else 0.0)
    return {'recall': recall, 'precision': precision, 'f1': f1, 'mean_iou': mean_iou}


# ── Initialise per-method running totals ─────────────────────────────────────
method_keys = ['A_rms', 'B_mfcc', 'C_wavlm', 'D_silero', 'union']
method_totals = {k: _empty_totals() for k in method_keys}
det_counts    = {k: [] for k in method_keys}
n_files_with_gt = 0

# ── Loop over files ───────────────────────────────────────────────────────────
json_files = sorted(glob.glob(os.path.join(input_dir, '*.json')))

if not json_files:
    print(f'No JSON metadata files found in {input_dir}. Upload audio + JSON first.')
else:
    print(f'Benchmarking {len(json_files)} file(s) ...\n')

for meta_file in json_files:
    with open(meta_file) as f:
        metadata = json.load(f)

    audio_path = os.path.join(input_dir, os.path.basename(metadata.get('output', '')))
    gt_events  = metadata.get('events', [])

    if not os.path.exists(audio_path):
        print(f'  SKIP {os.path.basename(audio_path)} — file not found')
        continue
    if not gt_events:
        print(f'  SKIP {os.path.basename(audio_path)} — no GT events in JSON')
        continue

    n_files_with_gt += 1
    print(f'  {os.path.basename(audio_path)}  ({len(gt_events)} GT event(s))')

    y, sr    = librosa.load(audio_path, sr=None, mono=True)
    duration = len(y) / sr

    # ── Run each method independently ─────────────────────────────────────────
    rms_raw   = detect_rms_dropouts(y, sr)
    mfcc_raw  = detect_mfcc_discontinuities(y, sr)
    wavlm_raw = (detect_wavlm_discontinuities(audio_path, verbose=False)
                 if wavlm_model is not None else [])
    sil_raw   = (detect_silero_dropouts(audio_path, verbose=False)
                 if silero_vad_model is not None else [])

    rms_det   = _pad_and_merge_single(rms_raw,   duration)
    mfcc_det  = _pad_and_merge_single(mfcc_raw,  duration)
    wavlm_det = _pad_and_merge_single(wavlm_raw, duration)
    sil_det   = _pad_and_merge_single(sil_raw,   duration)

    # Union: merge all four via the standard merge function, mode='any'
    union_merged = merge_events_with_source(
        rms_raw, mfcc_raw, wavlm_raw, sil_raw, audio_duration=duration)
    union_det = [{'start_sec': s, 'end_sec': e} for s, e, _ in union_merged]

    method_dets = {
        'A_rms':    rms_det,
        'B_mfcc':   mfcc_det,
        'C_wavlm':  wavlm_det,
        'D_silero': sil_det,
        'union':    union_det,
    }

    for key, det in method_dets.items():
        m = evaluate_detection(gt_events, det)
        _accumulate(method_totals[key], m)
        det_counts[key].append(m['n_detected'])

    print(f'    RMS:{len(rms_det)}  MFCC:{len(mfcc_det)}  '
          f'WavLM:{len(wavlm_det)}  Silero:{len(sil_det)}  Union:{len(union_det)}')

# ── Print comparison table ────────────────────────────────────────────────────
print(f'\n{"=" * 72}')
print(f'DETECTION BENCHMARK  --  {n_files_with_gt} file(s) with ground truth')
print(f'{"=" * 72}')
print(f'  IoU match threshold: {MATCH_IOU_THRESHOLD}')
print()

print(f'  {"Method":<18} {"Recall":>8} {"Prec":>8} {"F1":>8} {"MeanIoU":>9} {"Det/file":>9}')
print('  ' + '-' * 64)

labels = {
    'A_rms':    'A - RMS',
    'B_mfcc':   'B - MFCC',
    'C_wavlm':  'C - WavLM',
    'D_silero': 'D - Silero VAD',
    'union':    'Union (any)',
}
available = {
    'A_rms':    True,
    'B_mfcc':   True,
    'C_wavlm':  wavlm_model is not None,
    'D_silero': silero_vad_model is not None,
    'union':    True,
}

summaries = {k: _summary(method_totals[k]) for k in method_keys}
best_f1 = max(s['f1'] for s in summaries.values())

for key in method_keys:
    if not available[key]:
        print(f'  {labels[key]:<18} (model not loaded)')
        continue
    s = summaries[key]
    avg_det = (sum(det_counts[key]) / len(det_counts[key])
               if det_counts[key] else 0.0)
    marker = ' <- best F1' if abs(s['f1'] - best_f1) < 1e-6 and best_f1 > 0 else ''
    print(f'  {labels[key]:<18} '
          f'{s["recall"]:>8.3f} {s["precision"]:>8.3f} {s["f1"]:>8.3f} '
          f'{s["mean_iou"]:>9.3f} {avg_det:>9.1f}{marker}')

print(f'\n  GT events total: {method_totals["A_rms"]["n_gt"]}')
print('  Use the best-performing method as DETECTION_MODE in Step 7.')

# ── Save detection benchmark results to Drive ─────────────────────────────────
try:
    import json as _json
    from datetime import datetime as _dt
    _bench_ts  = _dt.now().strftime("%Y%m%d_%H%M%S")
    _bench_tag = RUN_TAG if "RUN_TAG" in vars() else "benchmark"
    _bench_data = {
        "run_tag":         _bench_tag,
        "timestamp":       _bench_ts,
        "n_files_with_gt": n_files_with_gt,
        "iou_threshold":   MATCH_IOU_THRESHOLD,
        "methods": {
            k: {
                **_summary(method_totals[k]),
                "avg_det_per_file": (
                    sum(det_counts[k]) / len(det_counts[k]) if det_counts[k] else 0.0
                ),
                "available": available.get(k, False),
            }
            for k in method_keys
        },
    }
    _bench_path = os.path.join(
        output_dir, f"{_bench_tag}_{_bench_ts}_detection_benchmark.json"
    )
    with open(_bench_path, "w") as _f:
        _json.dump(_bench_data, _f, indent=2)
    print(f"\nSaved: {os.path.basename(_bench_path)}")
except Exception as _e:
    print(f"Could not save benchmark report ({_e}).")

### Step 7: Run the Full Inpainting Pipeline

Loads PlayDiffusion and processes every sample in `input_dir`. Set `DETECTION_MODE` based on the Step 7a benchmark results.

Available modes: `any`, `rms`, `mfcc`, `wavlm`, `silero`, `both`, `wavlm+rms`, `wavlm+silero`

For each audio file it:
1. Transcribes with Qwen3-ASR (or Whisper) and corrects with Qwen3-32B via OpenRouter
2. Runs all detection methods and merges using the configured mode
3. If JSON events exist: evaluates detection vs ground truth; uses GT events for inpainting (or detected events if `USE_DETECTED_EVENTS = True`)
4. For each event: re-transcribes, snaps to word boundaries, then either crossfades (silence gaps) or calls PlayDiffusion to regenerate the region
5. Saves `<name>_inpainted.wav` and prints aggregate mel-MSE / STOI improvement

### Step 8: Experiment Configuration

Edit `EXPERIMENT_CONFIGS` below to define which pipeline variants to run.
The inpainting loop (Step 9) will process every audio file for **each config**
in sequence — one Colab session, one press of **Run All**.

**Config keys:**
- `run_tag` — label used in output filenames and eval reports *(required)*
- `detection_mode` — which detector(s) drive inpainting:
  `"wavlm+silero"` | `"silero"` | `"wavlm"` | `"rms"` | `"mfcc"` | `"any"` | `"both"`
- `use_detected_events` — `False`: inpaint the ground-truth cut regions (oracle ceiling);
  `True`: inpaint wherever the detector fired (fully autonomous pipeline)

**Tip:** Keep GT-events runs (`use_detected_events: False`) in the list to
establish the restoration ceiling independently of detection quality.
The detection benchmark (Step 7a) already covers detector comparison without
re-running PlayDiffusion, so the runs below should focus on end-to-end evaluation.

In [ ]:
# ── Experiment queue — edit this cell, then Run All ─────────────────────────
#
# events_source controls which event timestamps are used for inpainting:
#   "gt"            — ground-truth cut timestamps from the input JSON
#   "rms"           — RMS energy dropout detector only
#   "mfcc"          — MFCC discontinuity detector only
#   "wavlm"         — WavLM embedding discontinuity detector only
#   "silero"        — Silero VAD detector only
#   "both"          — RMS AND MFCC (intersection)
#   "wavlm+rms"     — WavLM AND RMS (intersection)
#   "wavlm+silero"  — WavLM AND Silero (intersection)
#   "any"           — union of all four detectors
#
# The pre-detection cell (Step 9) must run first — it caches detected events
# so detection runs once total, not once per experiment.
# ─────────────────────────────────────────────────────────────────────────────

EXPERIMENT_CONFIGS = [
    # ── No detector: ground-truth cut timestamps (oracle ceiling) ─────────────
    {
        "run_tag":      "no_detector",
        "events_source": "gt",
    },

    # ── Single-method detectors ───────────────────────────────────────────────
    {"run_tag": "auto_rms",    "events_source": "rms"},
    {"run_tag": "auto_mfcc",   "events_source": "mfcc"},
    {"run_tag": "auto_wavlm",  "events_source": "wavlm"},
    {"run_tag": "auto_silero", "events_source": "silero"},

    # ── Two-method combinations (intersection: both methods must fire) ─────────
    {"run_tag": "auto_rms_mfcc",     "events_source": "both"},
    {"run_tag": "auto_wavlm_rms",    "events_source": "wavlm+rms"},
    {"run_tag": "auto_wavlm_silero", "events_source": "wavlm+silero"},

    # ── Union: any detector fires ─────────────────────────────────────────────
    {"run_tag": "auto_any", "events_source": "any"},
]

# ── Preview ───────────────────────────────────────────────────────────────────
print(f"{len(EXPERIMENT_CONFIGS)} experiment(s) queued:\n")
for i, cfg in enumerate(EXPERIMENT_CONFIGS):
    print(f"  {i+1:2d}. {cfg['run_tag']:<25s}  events_source={cfg['events_source']}")


### Step 9: Pre-Detection — Cache All Detector Results

Runs all 4 detectors on every distorted file **once** and saves the results to
`output_dir/detected/B##_##.json`.  The inpainting loop (Step 10) reads from
this cache — detection is never repeated across the 9 experiments.

Re-run this cell if you change the distorted audio or want to refresh cached
results (they are skipped if the JSON already exists).

**Input data required in `input_dir` on Drive:**

| File | Contents |
|------|----------|
| `B##_##.wav` | Distorted audio (from `data/distorted/`) |
| `B##_##.json` | Ground-truth events (from `data/events/`) |

Format of the GT JSON:
```json
{ "output": "B01_01.wav", "events": [{"start_sec": 1.24, "end_sec": 1.38}] }
```


In [ ]:
# ── Step 9: Pre-Detection ─────────────────────────────────────────────────────
# Runs all detectors on every file once and caches results to Drive.
# The inpainting loop reads from this cache — detection won't re-run per config.
# ─────────────────────────────────────────────────────────────────────────────

import os, json, glob

DETECTED_DIR = os.path.join(output_dir, "detected")
os.makedirs(DETECTED_DIR, exist_ok=True)

DETECTION_MODES = [
    "rms", "mfcc", "wavlm", "silero",
    "both", "wavlm+rms", "wavlm+silero", "any",
]

_json_files = sorted(glob.glob(os.path.join(input_dir, "*.json")))
print(f"Pre-detecting events for {len(_json_files)} file(s)\n"
      f"Cache dir: {DETECTED_DIR}\n")

for _mf in _json_files:
    with open(_mf) as _f:
        _meta = json.load(_f)
    _audio = os.path.join(input_dir, os.path.basename(_meta.get("output", "")))
    _base  = os.path.splitext(os.path.basename(_audio))[0]
    _out   = os.path.join(DETECTED_DIR, f"{_base}.json")

    if os.path.exists(_out):
        print(f"  {_base}: cached — skipping (delete file to re-detect)")
        continue
    if not os.path.exists(_audio):
        print(f"  {_base}: audio not found — skipping")
        continue

    print(f"  {_base}:")
    _detectors = {}
    for _mode in DETECTION_MODES:
        _evs = detect_corrupted_regions(_audio, verbose=False, mode=_mode)
        _detectors[_mode] = [
            {"start_sec": e["start_sec"], "end_sec": e["end_sec"]} for e in _evs
        ]
        print(f"    {_mode:<16s}: {len(_evs)} event(s)")

    with open(_out, "w") as _f:
        json.dump({"file": os.path.basename(_audio), "detectors": _detectors},
                  _f, indent=2)

_cached = len(glob.glob(os.path.join(DETECTED_DIR, "*.json")))
print(f"\nDone. {_cached}/{len(_json_files)} file(s) cached in {DETECTED_DIR}")


In [ ]:
# ── Step 9: Run All Experiments ─────────────────────────────────────────────
#
# Loads PlayDiffusion once, then iterates over every config in
# EXPERIMENT_CONFIGS. For each config it:
#   1. Runs detection + optional evaluation against GT events
#   2. Inpaints each audio file
#   3. Computes audio quality metrics vs clean original (if available)
#   4. Saves run_config.json, audio_quality.json, detection_metrics.json
# ─────────────────────────────────────────────────────────────────────────────

import sys, os, json, glob
sys.path.insert(0, '/content/PlayDiffusion/src')

import numpy as np
import soundfile as sf
import librosa
import torchaudio
from playdiffusion import PlayDiffusion, InpaintInput
import scipy.io.wavfile as wav
from datetime import datetime as _dt

try:
    from pystoi import stoi as compute_stoi
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "pystoi"], check=True)
    from pystoi import stoi as compute_stoi

# ── Load PlayDiffusion once (reused for all experiments) ──────────────────────
print("Loading PlayDiffusion model ...")
inpainter = PlayDiffusion()
print("Model loaded.\n")

# ── Fixed settings (same for every experiment) ────────────────────────────────
CLEAN_AUDIO_DIR = "/content/drive/MyDrive/samples/raw_audio"
PAD_SEC         = 0.1
DEDUP_GAP_SEC   = 0.30
DETECTED_DIR    = os.path.join(output_dir, "detected")  # written by Step 9

# ── Audio quality helpers ─────────────────────────────────────────────────────
def _align_length(a, b):
    if len(b) < len(a):
        b = np.concatenate([b, np.zeros(len(a) - len(b), dtype=b.dtype)])
    elif len(b) > len(a):
        b = b[:len(a)]
    return b

def _to_mono_16k(y, sr):
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != 16000:
        y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=16000)
    return y.astype(np.float32)

def audio_similarity(reference_path, candidate_path):
    ref,  sr_ref  = sf.read(reference_path)
    cand, sr_cand = sf.read(candidate_path)
    ref  = _to_mono_16k(ref,  sr_ref)
    cand = _to_mono_16k(cand, sr_cand)
    cand = _align_length(ref, cand)
    mel_kw = dict(sr=16000, n_fft=512, hop_length=160, n_mels=64)
    mel_ref  = librosa.power_to_db(librosa.feature.melspectrogram(y=ref,  **mel_kw))
    mel_cand = librosa.power_to_db(librosa.feature.melspectrogram(y=cand, **mel_kw))
    mel_mse  = float(np.mean((mel_ref - mel_cand) ** 2))
    try:
        stoi_score = float(compute_stoi(ref, cand, 16000, extended=True))
    except Exception as _e:
        print(f"    STOI failed: {_e}")
        stoi_score = float("nan")
    return {"mel_mse": mel_mse, "stoi": stoi_score}

# ── Discover audio files once (same set for every experiment) ─────────────────
json_files = sorted(glob.glob(os.path.join(input_dir, "*.json")))
if not json_files:
    print(f"No JSON metadata files in {input_dir} — upload files first.")

# ═════════════════════════════════════════════════════════════════════════════
# EXPERIMENT LOOP
# ═════════════════════════════════════════════════════════════════════════════
for _cfg_idx, _cfg in enumerate(EXPERIMENT_CONFIGS):
    RUN_TAG       = _cfg["run_tag"]
    events_source = _cfg["events_source"]

    print(f"\n{'#' * 72}")
    print(f"# Experiment {_cfg_idx + 1}/{len(EXPERIMENT_CONFIGS)}: {RUN_TAG}")
    print(f"#   events_source = {events_source}")
    print(f"{'#' * 72}\n")

    # Per-experiment state
    eval_totals     = {"n_gt": 0, "n_detected": 0, "n_matched": 0,
                       "iou_sum": 0.0, "iou_count": 0}
    quality_results = []

    # ── Process each audio file ───────────────────────────────────────────────
    for meta_file in json_files:
        with open(meta_file, 'r') as f:
            metadata = json.load(f)

        audio_path = os.path.join(input_dir,
                                   os.path.basename(metadata.get("output", "")))
        base       = os.path.splitext(os.path.basename(audio_path))[0]
        clean_path = os.path.join(CLEAN_AUDIO_DIR, f"{base}.wav")

        print(f"{'=' * 60}")
        print(f"Processing: {os.path.basename(audio_path)}")

        if not os.path.exists(audio_path):
            print("  Audio file not found — skipping.\n")
            continue

        # Transcribe + LLM correction
        corrected_text, original_word_timestamps = \
            get_corrected_text_and_original_word_times(audio_path)

        # Load pre-computed detected events (Step 9 must have run first)
        gt_events = metadata.get("events", [])
        _det_file = os.path.join(DETECTED_DIR, f"{base}.json")
        if os.path.exists(_det_file):
            with open(_det_file) as _f:
                _all_det = json.load(_f).get("detectors", {})
            if events_source == "gt":
                # Use "any" union for detection metrics; GT events for inpainting
                detected_events = [
                    {"start_sec": e["start_sec"], "end_sec": e["end_sec"]}
                    for e in _all_det.get("any", [])
                ]
                events = gt_events if gt_events else detected_events
                print(f"  Using {len(events)} GT event(s) for inpainting.")
            else:
                detected_events = [
                    {"start_sec": e["start_sec"], "end_sec": e["end_sec"]}
                    for e in _all_det.get(events_source, [])
                ]
                events = detected_events
                print(f"  Using {len(events)} detected event(s) "
                      f"[{events_source}] for inpainting.")
        else:
            # Fallback: run on-the-fly if cache missing
            _mode = "any" if events_source == "gt" else events_source
            print(f"  [WARN] Pre-detection cache missing — "
                  f"detecting on-the-fly (mode={_mode})")
            detected_events = detect_corrupted_regions(
                audio_path, verbose=True, mode=_mode)
            events = (gt_events if gt_events else detected_events
                      if events_source == "gt" else detected_events)

        if gt_events:
            metrics = evaluate_detection(gt_events, detected_events)
            print(f"  [Detection] GT: {metrics['n_gt']}  "
                  f"Detected: {metrics['n_detected']}  "
                  f"Matched: {metrics['n_matched']}")
            print(f"  [Detection] Recall: {metrics['recall']:.2%}  "
                  f"Precision: {metrics['precision']:.2%}  "
                  f"Mean IoU: {metrics['mean_iou']:.3f}")
            eval_totals["n_gt"]       += metrics["n_gt"]
            eval_totals["n_detected"] += metrics["n_detected"]
            eval_totals["n_matched"]  += metrics["n_matched"]
            eval_totals["iou_sum"]    += metrics["mean_iou"] * metrics["n_matched"]
            eval_totals["iou_count"]  += metrics["n_matched"]

        # Deduplicate overlapping events
        if len(events) > 1:
            sorted_ev = sorted(events, key=lambda e: e["start_sec"])
            deduped   = [dict(sorted_ev[0])]
            for e in sorted_ev[1:]:
                if e["start_sec"] - deduped[-1]["end_sec"] <= DEDUP_GAP_SEC:
                    deduped[-1]["end_sec"] = max(deduped[-1]["end_sec"],
                                                  e["end_sec"])
                else:
                    deduped.append(dict(e))
            if len(deduped) < len(events):
                print(f"  [DEDUP] {len(events)} → {len(deduped)} events")
            events = deduped

        out_file = os.path.join(output_dir, f"{base}_inpainted.wav")

        if not events:
            print("  No events — copying input as output.")
            y, sr = sf.read(audio_path)
            sf.write(out_file, y, sr)
        else:
            temp_path = os.path.join(output_dir, f"{base}_temp.wav")

            # Pad temp file so PlayDiffusion has context near edges
            y, sr = sf.read(audio_path)
            pad   = np.zeros(int(PAD_SEC * sr), dtype=y.dtype)
            y_pad = (np.concatenate([pad, y, pad]) if y.ndim == 1
                     else np.concatenate([
                         np.zeros((int(PAD_SEC * sr), y.shape[1]), dtype=y.dtype),
                         y,
                         np.zeros((int(PAD_SEC * sr), y.shape[1]), dtype=y.dtype)]))
            sf.write(temp_path, y_pad, sr)

            events_padded = [
                {"start_sec": e["start_sec"] + PAD_SEC,
                 "end_sec":   e["end_sec"]   + PAD_SEC}
                for e in events
            ]

            for i, event in enumerate(events_padded):
                t0 = event["start_sec"]
                t1 = event["end_sec"]

                cur_transcript, cur_word_ts = transcribe_with_timestamps(temp_path)

                overlapping = [
                    (idx, w) for idx, w in enumerate(cur_word_ts)
                    if w['start'] < t1 and w['end'] > t0
                ]

                if overlapping:
                    ov_idx, ov_w = zip(*overlapping)
                    adj_t0   = min(w['start'] for w in ov_w)
                    adj_t1   = max(w['end']   for w in ov_w)
                    silence  = False
                else:
                    adj_t0, adj_t1 = t0, t1
                    silence        = True

                print(f"  Event {i+1}: {t0:.3f}s–{t1:.3f}s  "
                      f"→ adjusted {adj_t0:.3f}s–{adj_t1:.3f}s")

                if silence:
                    print("  → Silence gap: crossfade.")
                    smooth_silence_gap(temp_path, adj_t0, adj_t1, temp_path)
                    continue

                if has_acoustic_discontinuity(temp_path, t0):
                    keep = set(range(len(cur_word_ts))) - set(ov_idx)
                    input_txt = ' '.join(
                        cur_word_ts[k]['word'] for k in sorted(keep))
                    print(f"  → Acoustic artifact: forcing text diff.")
                else:
                    input_txt = cur_transcript

                try:
                    request = InpaintInput(
                        audio=temp_path,
                        input_text=input_txt,
                        output_text=corrected_text,
                        start_time=adj_t0,
                        end_time=adj_t1,
                        input_word_times=cur_word_ts,
                    )
                    sample_rate, audio_data = inpainter.inpaint(request)
                    wav.write(temp_path, sample_rate, audio_data)
                except Exception as _e:
                    print(f"  FAILED: {_e}")

            # Trim padding and save
            y_final, sr_final = sf.read(temp_path)
            pad_s = int(PAD_SEC * sr_final)
            y_final = (y_final[pad_s:-pad_s] if y_final.ndim == 1
                       else y_final[pad_s:-pad_s, :])
            sf.write(out_file, y_final, sr_final)
            os.remove(temp_path)
            print(f"  Saved: {out_file}")

        # Audio quality vs clean original
        if os.path.exists(clean_path):
            sim_c = audio_similarity(clean_path, audio_path)
            sim_i = audio_similarity(clean_path, out_file)
            mel_l  = sim_c['mel_mse'] - sim_i['mel_mse']
            stoi_l = sim_i['stoi']    - sim_c['stoi']
            print(f"    Corrupted : mel={sim_c['mel_mse']:.2f}  stoi={sim_c['stoi']:.3f}")
            print(f"    Inpainted : mel={sim_i['mel_mse']:.2f}  stoi={sim_i['stoi']:.3f}")
            print(f"    Lift      : mel={mel_l:+.2f}  stoi={stoi_l:+.3f}")
            quality_results.append({
                "sample":        base,
                "n_events":      len(events),
                "mel_corrupt":   sim_c["mel_mse"],
                "mel_inpainted": sim_i["mel_mse"],
                "stoi_corrupt":  sim_c["stoi"],
                "stoi_inpainted":sim_i["stoi"],
            })
        else:
            print(f"  [WARN] {clean_path} not found — skipping quality eval.")
        print()

    # ── Per-experiment summary ────────────────────────────────────────────────
    print(f"\n{'=' * 60}")
    print(f"DETECTION METRICS — {RUN_TAG}")
    print(f"{'=' * 60}")
    print_aggregate_metrics(eval_totals)

    if quality_results:
        _n = len(quality_results)
        avg_mel  = sum(r['mel_corrupt']   - r['mel_inpainted']  for r in quality_results) / _n
        avg_stoi = sum(r['stoi_inpainted'] - r['stoi_corrupt']  for r in quality_results) / _n
        n_better = sum(1 for r in quality_results
                       if r['mel_corrupt'] > r['mel_inpainted']
                       and r['stoi_inpainted'] > r['stoi_corrupt'])
        n_worse  = sum(1 for r in quality_results
                       if r['mel_corrupt'] < r['mel_inpainted']
                       and r['stoi_inpainted'] < r['stoi_corrupt'])
        print(f"\nAUDIO QUALITY — {RUN_TAG}")
        print(f"  Avg mel_mse lift: {avg_mel:+.2f}  |  Avg STOI lift: {avg_stoi:+.3f}")
        print(f"  Improved both: {n_better}/{_n}  |  Worse both: {n_worse}/{_n}")

    # ── Save reports to Drive ─────────────────────────────────────────────────
    _ts   = _dt.now().strftime("%Y%m%d_%H%M%S")
    _stem = f"{RUN_TAG}_{_ts}"

    _run_cfg = {
        "run_tag":             RUN_TAG,
        "timestamp":           _ts,
        "notebook_version":    "v5.0",
        "events_source":       events_source,
        "asr_model":           "qwen3-asr" if qwen_asr_model is not None
                                else "whisper-base",
        "llm_correction":      OPENROUTER_API_KEY is not None,
    }
    with open(os.path.join(output_dir, f"{_stem}_run_config.json"), "w") as _f:
        json.dump(_run_cfg, _f, indent=2)

    if quality_results:
        _n = len(quality_results)
        _aq = {
            "run_tag":   RUN_TAG,
            "timestamp": _ts,
            "per_file":  quality_results,
            "aggregate": {
                "n":               _n,
                "avg_mel_lift":    sum(r['mel_corrupt']    - r['mel_inpainted']
                                        for r in quality_results) / _n,
                "avg_stoi_lift":   sum(r['stoi_inpainted'] - r['stoi_corrupt']
                                        for r in quality_results) / _n,
                "n_improved_both": sum(1 for r in quality_results
                                        if r['mel_corrupt']    > r['mel_inpainted']
                                        and r['stoi_inpainted'] > r['stoi_corrupt']),
                "n_worse_both":    sum(1 for r in quality_results
                                        if r['mel_corrupt']    < r['mel_inpainted']
                                        and r['stoi_inpainted'] < r['stoi_corrupt']),
            },
        }
        with open(os.path.join(output_dir, f"{_stem}_audio_quality.json"), "w") as _f:
            json.dump(_aq, _f, indent=2)

    _det = {
        "run_tag":        RUN_TAG,
        "timestamp":      _ts,
        "events_source":  events_source,
        "iou_threshold":  MATCH_IOU_THRESHOLD,
        "aggregate": {
            "n_gt":       eval_totals["n_gt"],
            "n_detected": eval_totals["n_detected"],
            "n_matched":  eval_totals["n_matched"],
            "recall":     (eval_totals["n_matched"] / eval_totals["n_gt"]
                            if eval_totals["n_gt"] else 0.0),
            "precision":  (eval_totals["n_matched"] / eval_totals["n_detected"]
                            if eval_totals["n_detected"] else 0.0),
            "mean_iou":   (eval_totals["iou_sum"] / eval_totals["iou_count"]
                            if eval_totals["iou_count"] else 0.0),
        },
    }
    with open(os.path.join(output_dir, f"{_stem}_detection_metrics.json"), "w") as _f:
        json.dump(_det, _f, indent=2)

    print(f"\nSaved: {_stem}_run_config.json")
    if quality_results:
        print(f"Saved: {_stem}_audio_quality.json")
    print(f"Saved: {_stem}_detection_metrics.json")
    print(f"\nNext: python evals/eval_runner.py "
          f"--restored-dir data/restored_{RUN_TAG}/ --label {RUN_TAG}")

# ═══════════════════════════════════════════════════════════════════════════
print(f"\n{'#' * 72}")
print(f"# All {len(EXPERIMENT_CONFIGS)} experiment(s) complete.")
print(f"# Run: python evals/compare_runs.py results/summary_*.json")
print(f"{'#' * 72}")


In [ ]:
# This clears the output folder !
# import glob, os

# for f in glob.glob(os.path.join(output_dir, "*")):
#     os.remove(f)

# print("Output folder cleared.")